In [23]:
import datetime as dt
import sys
from pathlib import Path

# Add the workspace root to Python path so we can import from src
workspace_root = (
    Path(__file__).parent.parent.parent
    if "__file__" in globals()
    else Path.cwd().parent.parent
)
sys.path.insert(0, str(workspace_root))

from dask.diagnostics import ProgressBar
import dask.dataframe as dd
import pandas as pd
import numpy as np
from sqlalchemy import select, create_engine
from sqlalchemy.sql.expression import func
from sqlalchemy.sql.expression import literal_column, literal
from sqlalchemy.dialects.postgresql import INTERVAL
from dotenv import load_dotenv
import os
from statsmodels.tsa.stattools import coint
import mc_postgres_db.models as models
from sqlalchemy.orm import Session
from coiled import Cluster
from dask import delayed
from dask.distributed import LocalCluster
from src.utils.stochastic_models import OrnsteinUhlenbeck
import statsmodels.api as sm
from mc_postgres_db.operations import set_data

load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

CLUSTER_TYPE = "coiled"
N_WORKERS = 20

engine = create_engine(POSTGRES_URL)

In [ ]:
cluster = None
if CLUSTER_TYPE == "local":
    try:
        cluster.close()
    except:
        pass
    cluster = LocalCluster(
        name="local-cluster", n_workers=N_WORKERS, memory_limit="3GB"
    )
elif CLUSTER_TYPE == "coiled":
    cluster = Cluster(
        name="prefect-cluster",
        n_workers=N_WORKERS,
        region="us-east-1",
        container="ghcr.io/manning-capital/mc-notebooks:main",
        worker_memory="16GB",
        worker_cpu=2,
    )

[2025-11-03 17:50:52,411][INFO    ][coiled] Creating software environment
[2025-11-03 17:50:52,563][INFO    ][coiled] Software environment created
[2025-11-03 17:50:53,489][INFO    ][coiled] Creating Cluster (name: prefect-cluster, https://cloud.coiled.io/clusters/1242425 ). This usually takes 1-2 minutes...


In [3]:
client = cluster.get_client()
display(client)

<Client: 'tls://10.0.207.159:8786' processes=19 threads=38, memory=130.15 GiB>

In [4]:
max_groups = 5000
lookback_days = 30

In [5]:
date: dt.date = dt.datetime.now(dt.timezone.utc).date() - dt.timedelta(days=1)
end = dt.datetime.combine(date, dt.time.min)
start = end - dt.timedelta(days=lookback_days)
start_naive = start.replace(tzinfo=None).replace(second=0, microsecond=0)
end_naive = end.replace(tzinfo=None).replace(second=0, microsecond=0)
print(f"Start: {start}, End: {end}")

Start: 2025-10-03 00:00:00, End: 2025-11-02 00:00:00


In [6]:
with Session(engine) as session:
    # Get all provider asset group id(s)
    provider_asset_group_ids = session.scalars(
        select(models.ProviderAssetGroup.id).limit(max_groups)
    ).all()
print(
    f"Provider asset group ids (count: {len(provider_asset_group_ids)}): {provider_asset_group_ids}"
)

Provider asset group ids (count: 4184): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 21

In [7]:
@delayed
def load_pairs_trading_frame_chunk(
    provider_asset_group_ids: list[int],
    start: dt.datetime,
    end: dt.datetime,
    conn_string: str,
    market_data: pd.DataFrame,
) -> pd.DataFrame:
    """
    Load the pairs trading frame for a chunk of provider asset groups.
    Returns only the essential columns needed for cointegration analysis.

    Args:
        provider_asset_group_ids: List of provider asset group IDs to process
        start: Start datetime (timezone-naive)
        end: End datetime (timezone-naive)
        conn_string: Database connection string

    Returns:
        pandas DataFrame indexed by provider_asset_group_id with columns:
            - timestamp
            - close_1
            - close_2
    """
    engine = create_engine(conn_string)

    # Step 1: Generate timeframe
    start_str = start.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")
    end_str = end.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")
    time_frame = pd.read_sql(
        select(
            func.generate_series(
                literal_column(start_str),
                literal_column(end_str),
                func.cast(literal("1 minute"), INTERVAL),
            ).label("timestamp")
        ),
        engine,
    )

    # Step 2: Load provider asset group members
    members = pd.read_sql(
        select(
            models.ProviderAssetGroupMember.provider_asset_group_id,
            models.ProviderAssetGroupMember.order,
            models.ProviderAssetGroupMember.provider_id,
            models.ProviderAssetGroupMember.from_asset_id,
            models.ProviderAssetGroupMember.to_asset_id,
        ).where(
            models.ProviderAssetGroupMember.provider_asset_group_id.in_(
                provider_asset_group_ids
            )
        ),
        engine,
    )

    # Step 3: Cross join
    time_frame["key"] = 1
    members["key"] = 1
    full_frame = time_frame.merge(members, on="key").drop(columns=["key"])
    full_frame = full_frame.sort_values("timestamp")

    # Step 5: Merge_asof
    full_market_frame = pd.merge_asof(
        full_frame,
        market_data,
        on="timestamp",
        by=["provider_id", "from_asset_id", "to_asset_id"],
        direction="backward",
    )

    # Step 6: Split by order and create pairs - only keep essential columns
    close_1 = full_market_frame[full_market_frame["order"] == 1][
        ["timestamp", "provider_asset_group_id", "close"]
    ].rename(columns={"close": "close_1"})
    close_2 = full_market_frame[full_market_frame["order"] == 2][
        ["timestamp", "provider_asset_group_id", "close"]
    ].rename(columns={"close": "close_2"})

    # Merge to create pairs - only timestamp, close_1, close_2
    pairs = pd.merge(
        close_1, close_2, on=["timestamp", "provider_asset_group_id"], how="inner"
    )

    # Keep only essential columns
    pairs = pairs[["provider_asset_group_id", "timestamp", "close_1", "close_2"]]

    # Set index to provider_asset_group_id
    pairs = pairs.set_index("provider_asset_group_id")

    engine.dispose()
    return pairs


def get_pairs_trading_frame(
    start: dt.datetime,
    end: dt.datetime,
    provider_asset_group_ids: list[int],
    conn_string: str,
    n_workers: int = 10,
) -> dd.DataFrame:
    """
    Get the pairs trading frame with only essential columns for cointegration analysis.

    Returns:
        Dask DataFrame indexed by provider_asset_group_id with columns:
            - timestamp
            - close_1
            - close_2
    """
    # Split provider asset groups into chunks
    n_chunks = min(n_workers, len(provider_asset_group_ids))
    group_chunks = np.array_split(provider_asset_group_ids, n_chunks)

    # Load market data once
    market_data = pd.read_sql(
        select(
            models.ProviderAssetMarket.timestamp,
            models.ProviderAssetMarket.provider_id,
            models.ProviderAssetMarket.from_asset_id,
            models.ProviderAssetMarket.to_asset_id,
            models.ProviderAssetMarket.close,
        )
        .where(models.ProviderAssetMarket.timestamp.between(start_naive, end_naive))
        .order_by(models.ProviderAssetMarket.timestamp),
        engine,
    )

    # Broadcast to all workers as a shared future
    market_data_future = client.scatter(market_data, broadcast=True)

    # Create delayed tasks
    delayed_dfs = [
        load_pairs_trading_frame_chunk(
            chunk.tolist(), start, end, conn_string, market_data_future
        )
        for chunk in group_chunks
    ]

    # Define minimal schema
    meta = pd.DataFrame(
        {
            "timestamp": pd.Series(dtype="datetime64[ns]"),
            "close_1": pd.Series(dtype="float64"),
            "close_2": pd.Series(dtype="float64"),
        }
    )
    meta.index = pd.Index([], name="provider_asset_group_id", dtype="int64")

    # Convert to Dask DataFrame
    pairs_trading_frame = dd.from_delayed(delayed_dfs, meta=meta)

    # Set index to provider_asset_group_id
    pairs_trading_frame = pairs_trading_frame.set_index(
        "provider_asset_group_id", sorted=True
    )

    return pairs_trading_frame

In [8]:
# @delayed
# def load_provider_group_members_chunk(
#     group_ids: list[int], conn_string: str
# ) -> pd.DataFrame:
#     """
#     Load provider asset group members for a list of group_ids.
#     """
#     engine = create_engine(conn_string)
#     df = pd.read_sql(
#         select(
#             models.ProviderAssetGroupMember.provider_asset_group_id,
#             models.ProviderAssetGroupMember.order,
#             models.ProviderAssetGroupMember.provider_id,
#             models.ProviderAssetGroupMember.from_asset_id,
#             models.ProviderAssetGroupMember.to_asset_id,
#         ).where(models.ProviderAssetGroupMember.provider_asset_group_id.in_(group_ids)),
#         engine,
#     )
#     engine.dispose()
#     return df


# @delayed
# def load_market_data_chunk(start_time, end_time, conn_string: str) -> pd.DataFrame:
#     """
#     Load a chunk of market data for a time range.
#     """
#     engine = create_engine(conn_string)
#     df = pd.read_sql(
#         select(
#             models.ProviderAssetMarket.timestamp,
#             models.ProviderAssetMarket.provider_id,
#             models.ProviderAssetMarket.from_asset_id,
#             models.ProviderAssetMarket.to_asset_id,
#             models.ProviderAssetMarket.close,
#         )
#         .where(models.ProviderAssetMarket.timestamp.between(start_time, end_time))
#         .order_by(models.ProviderAssetMarket.timestamp),
#         engine,
#         index_col="timestamp",
#     )
#     engine.dispose()
#     return df.reset_index()


# def get_pairs_trading_frame(
#     start: dt.datetime,
#     end: dt.datetime,
#     provider_asset_group_ids: list[int],
#     conn_string: str,
#     n_workers: int = 10,
# ) -> dd.DataFrame:
#     """
#     Get the pairs trading frame for given parameters.

#     This function:
#     1. Generates a complete timeframe (1-minute intervals)
#     2. Loads provider asset group members in parallel
#     3. Creates a cross join to get all timestamp-member combinations
#     4. Loads market data in parallel
#     5. Performs merge_asof to join market prices
#     6. Splits by order and creates pairs (order 1 vs order 2)

#     Args:
#         start: Start datetime (timezone-naive)
#         end: End datetime (timezone-naive)
#         provider_asset_group_ids: List of provider asset group IDs to process
#         conn_string: Database connection string
#         n_workers: Number of parallel workers for loading data

#     Returns:
#         Dask DataFrame with columns:
#             - timestamp (index)
#             - provider_asset_group_id
#             - provider_id_1, from_asset_id_1, to_asset_id_1, close_1
#             - provider_id_2, from_asset_id_2, to_asset_id_2, close_2
#     """
#     # Step 1: Generate timeframe
#     start_str = start.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")
#     end_str = end.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")

#     time_frame = dd.read_sql_query(
#         select(
#             select(
#                 func.generate_series(
#                     literal_column(start_str),
#                     literal_column(end_str),
#                     func.cast(literal("1 minute"), INTERVAL),
#                 ).label("timestamp")
#             ).subquery("time_frame")
#         ),
#         conn_string,
#         index_col="timestamp",
#         bytes_per_chunk="512 MiB",
#     )
#     time_frame = time_frame.reset_index()

#     # Step 2: Load provider asset group members (in parallel)
#     n_partitions = min(n_workers, len(provider_asset_group_ids))
#     group_chunks = np.array_split(provider_asset_group_ids, n_partitions)

#     delayed_member_dfs = [
#         load_provider_group_members_chunk(chunk.tolist(), conn_string)
#         for chunk in group_chunks
#     ]

#     meta_members = pd.DataFrame(
#         {
#             "provider_asset_group_id": pd.Series(dtype="int64"),
#             "order": pd.Series(dtype="int64"),
#             "provider_id": pd.Series(dtype="int64"),
#             "from_asset_id": pd.Series(dtype="int64"),
#             "to_asset_id": pd.Series(dtype="int64"),
#         }
#     )

#     provider_asset_group_members = dd.from_delayed(
#         delayed_member_dfs, meta=meta_members
#     )

#     # Step 3: Cross join timeframe with members
#     time_frame["key"] = 1
#     provider_asset_group_members["key"] = 1
#     full_frame = time_frame.merge(provider_asset_group_members, on="key")
#     full_frame = full_frame.drop(columns=["key"])
#     full_frame = full_frame.sort_values(by="timestamp")
#     full_frame = full_frame.set_index("timestamp")

#     # Step 4: Load market data (in parallel)
#     time_chunks = pd.date_range(start, end, periods=n_workers + 1)

#     delayed_market_dfs = [
#         load_market_data_chunk(time_chunks[i], time_chunks[i + 1], conn_string)
#         for i in range(len(time_chunks) - 1)
#     ]

#     meta_market = pd.DataFrame(
#         {
#             "timestamp": pd.Series(dtype="datetime64[ns]"),
#             "provider_id": pd.Series(dtype="int64"),
#             "from_asset_id": pd.Series(dtype="int64"),
#             "to_asset_id": pd.Series(dtype="int64"),
#             "close": pd.Series(dtype="float64"),
#         }
#     )

#     market_data = dd.from_delayed(delayed_market_dfs, meta=meta_market)
#     market_data = market_data.sort_values(by="timestamp")
#     market_data = market_data.set_index("timestamp")

#     # Step 5: Merge_asof to join market prices
#     full_market_frame = dd.merge_asof(
#         full_frame,
#         market_data,
#         left_index=True,
#         right_index=True,
#         by=["provider_id", "from_asset_id", "to_asset_id"],
#     )

#     # Step 6: Split by order and create pairs
#     close_1 = full_market_frame.loc[
#         full_frame["order"] == 1,
#         [
#             "provider_asset_group_id",
#             "provider_id",
#             "from_asset_id",
#             "to_asset_id",
#             "close",
#         ],
#     ]
#     close_2 = full_market_frame.loc[
#         full_frame["order"] == 2,
#         [
#             "provider_asset_group_id",
#             "provider_id",
#             "from_asset_id",
#             "to_asset_id",
#             "close",
#         ],
#     ]

#     pairs_trading_frame = dd.merge(
#         close_1,
#         close_2,
#         on=["timestamp", "provider_asset_group_id"],
#         how="inner",
#         suffixes=("_1", "_2"),
#     )

#     return pairs_trading_frame

In [9]:
pairs_trading_frame = get_pairs_trading_frame(
    start_naive,
    end_naive,
    provider_asset_group_ids,
    engine.url.render_as_string(hide_password=False),
    N_WORKERS,
)

In [10]:
cointegration_p_values = pairs_trading_frame.groupby("provider_asset_group_id")[
    ["close_1", "close_2"]
].apply(
    lambda df: pd.Series(coint(df["close_1"], df["close_2"])[1], index=["p_value"]),
    meta={"p_value": pd.Series([], dtype=float)},
)

In [11]:
with ProgressBar():
    cointegration_p_values_computed = cointegration_p_values.compute()
cointegration_p_values_computed

,p_value
provider_asset_group_id,
20,4.930423e-01
33,2.150716e-01
54,2.254410e-03
88,7.482652e-01
104,7.893909e-08
...,...
4080,7.868368e-01
4098,1.041209e-01
4160,4.840063e-01


In [12]:
cointegrated_provider_asset_group_ids = cointegration_p_values_computed.loc[
    cointegration_p_values_computed["p_value"] < 0.001
].index.tolist()
print(
    f"Cointegrated provider asset group ids (count: {len(cointegrated_provider_asset_group_ids)}): {cointegrated_provider_asset_group_ids}"
)

Cointegrated provider asset group ids (count: 1129): [104, 154, 168, 180, 186, 295, 320, 323, 371, 419, 427, 599, 662, 680, 830, 866, 935, 1002, 1164, 1320, 1420, 1463, 1510, 1520, 1544, 1563, 1673, 1698, 1728, 1749, 1870, 1888, 2040, 2277, 2295, 2394, 2448, 2583, 2603, 2644, 2655, 2673, 2728, 2743, 2778, 3018, 3363, 3371, 3397, 3549, 3561, 3597, 3652, 3732, 3746, 3749, 3790, 3878, 3958, 4076, 4095, 99, 151, 264, 306, 674, 690, 700, 827, 892, 952, 953, 1112, 1235, 1319, 1324, 1364, 1384, 1402, 1436, 1509, 1546, 1737, 1892, 1996, 2096, 2101, 2296, 2299, 2325, 2444, 2552, 2595, 2616, 2710, 2734, 2757, 2777, 2887, 2998, 3013, 3021, 3105, 3212, 3215, 3337, 3447, 3486, 3497, 3607, 3635, 3764, 3768, 3802, 3871, 4000, 4017, 4064, 4093, 111, 200, 213, 232, 234, 390, 391, 428, 540, 940, 948, 1088, 1136, 1142, 1156, 1168, 1192, 1224, 1371, 1433, 1468, 1575, 1606, 1647, 1803, 1978, 2059, 2076, 2097, 2283, 2604, 2611, 2638, 2679, 2744, 2754, 2897, 2953, 3016, 3022, 3091, 3173, 3365, 3504, 3533, 36

In [13]:
def get_cointegrated_stats(df: pd.DataFrame) -> pd.Series:
    """
    Get the cointegrated stats for a given dataframe.
    """

    # Compute the linear regression.
    X = df["close_1"].to_numpy()
    y = df["close_2"].to_numpy()
    X = sm.add_constant(X)
    model = sm.OLS(y, X)
    results = model.fit()

    # Get the residuals.
    linear_fit_alpha = results.params[0]
    linear_fit_beta = results.params[1]
    linear_fit_mse = results.mse_total
    linear_fit_r_squared = results.rsquared
    linear_fit_r_squared_adj = results.rsquared_adj
    residuals = results.resid

    # Get the cointegration stats.
    ou_params = OrnsteinUhlenbeck().fit(residuals)

    return pd.Series(
        [
            linear_fit_alpha,
            linear_fit_beta,
            linear_fit_mse,
            linear_fit_r_squared,
            linear_fit_r_squared_adj,
            ou_params.mu,
            ou_params.theta,
            ou_params.sigma,
        ],
        index=[
            "linear_fit_alpha",
            "linear_fit_beta",
            "linear_fit_mse",
            "linear_fit_r_squared",
            "linear_fit_r_squared_adj",
            "ou_mu",
            "ou_theta",
            "ou_sigma",
        ],
        dtype=float,
    )

In [14]:
cointegrated_pairs_trading_frame = get_pairs_trading_frame(
    start_naive,
    end_naive,
    cointegrated_provider_asset_group_ids,
    engine.url.render_as_string(hide_password=False),
    N_WORKERS,
)

In [15]:
cointegrated_pairs_trading_stats = cointegrated_pairs_trading_frame.groupby(
    "provider_asset_group_id"
)[["close_1", "close_2"]].apply(
    lambda df: get_cointegrated_stats(df),
    meta={
        "linear_fit_alpha": pd.Series([], dtype=float),
        "linear_fit_beta": pd.Series([], dtype=float),
        "linear_fit_mse": pd.Series([], dtype=float),
        "linear_fit_r_squared": pd.Series([], dtype=float),
        "linear_fit_r_squared_adj": pd.Series([], dtype=float),
        "ou_mu": pd.Series([], dtype=float),
        "ou_theta": pd.Series([], dtype=float),
        "ou_sigma": pd.Series([], dtype=float),
    },
)

In [16]:
cointegrated_pairs_trading_stats_computed = cointegrated_pairs_trading_stats.compute()
cointegrated_pairs_trading_stats_computed

,linear_fit_alpha,linear_fit_beta,linear_fit_mse,linear_fit_r_squared,linear_fit_r_squared_adj,ou_mu,ou_theta,ou_sigma
provider_asset_group_id,,,,,,,,
104,-182.608047,185.028174,1.061809e-01,0.042425,0.042403,0.000436,-3.473592e-02,9.162937e-03
154,-84.336020,86.493453,2.149954e-01,0.025657,0.025634,0.001847,-1.710512e-02,2.752155e-02
168,-0.019992,0.019786,2.619634e-03,0.987708,0.987708,0.041125,-7.584455e-06,1.627671e-03
180,107.767677,-105.671015,2.149954e-01,0.010802,0.010779,0.001382,-2.346643e-02,2.388461e-02
186,0.000003,0.000012,1.037953e-12,0.984562,0.984562,0.024769,4.143518e-10,2.824289e-08
...,...,...,...,...,...,...,...,...
3541,-0.194807,0.228621,1.031218e-02,0.962595,0.962594,0.008047,8.908494e-05,2.481938e-03
3756,-0.324767,0.381641,2.843921e-02,0.972646,0.972646,0.009135,2.016150e-05,3.771759e-03
3976,-0.084006,0.616629,2.897839e-03,0.990929,0.990929,0.030446,9.802746e-06,1.266775e-03


In [19]:
cointegration_p_values_computed.to_csv("cointegration_p_values.csv")
cointegrated_pairs_trading_stats_computed.to_csv("cointegrated_pairs_trading_stats.csv")
cointegration_p_values_computed.to_parquet("cointegration_p_values.parquet")
cointegrated_pairs_trading_stats_computed.to_parquet(
    "cointegrated_pairs_trading_stats.parquet"
)

In [29]:
toset = cointegration_p_values_computed.merge(
    cointegrated_pairs_trading_stats_computed, left_index=True, right_index=True
).reset_index()
toset = toset.rename(columns={"p_value": "cointegration_p_value"})
toset["lookback_window_seconds"] = 30 * 24 * 60 * 60
toset["timestamp"] = end_naive
toset = toset[
    [
        "timestamp",
        "provider_asset_group_id",
        "lookback_window_seconds",
        "cointegration_p_value",
        "linear_fit_alpha",
        "linear_fit_beta",
        "linear_fit_mse",
        "linear_fit_r_squared",
        "linear_fit_r_squared_adj",
        "ou_mu",
        "ou_theta",
        "ou_sigma",
    ]
]
toset

,timestamp,provider_asset_group_id,lookback_window_seconds,cointegration_p_value,linear_fit_alpha,linear_fit_beta,linear_fit_mse,linear_fit_r_squared,linear_fit_r_squared_adj,ou_mu,ou_theta,ou_sigma
0,2025-11-02,104,2592000,7.893909e-08,-182.608047,185.028174,1.061809e-01,0.042425,0.042403,0.000436,-3.473592e-02,9.162937e-03
1,2025-11-02,154,2592000,5.787079e-28,-84.336020,86.493453,2.149954e-01,0.025657,0.025634,0.001847,-1.710512e-02,2.752155e-02
2,2025-11-02,168,2592000,1.443843e-11,-0.019992,0.019786,2.619634e-03,0.987708,0.987708,0.041125,-7.584455e-06,1.627671e-03
3,2025-11-02,180,2592000,0.000000e+00,107.767677,-105.671015,2.149954e-01,0.010802,0.010779,0.001382,-2.346643e-02,2.388461e-02
4,2025-11-02,186,2592000,5.754969e-05,0.000003,0.000012,1.037953e-12,0.984562,0.984562,0.024769,4.143518e-10,2.824289e-08
...,...,...,...,...,...,...,...,...,...,...,...,...
1124,2025-11-02,3541,2592000,5.422542e-04,-0.194807,0.228621,1.031218e-02,0.962595,0.962594,0.008047,8.908494e-05,2.481938e-03
1125,2025-11-02,3756,2592000,9.125116e-07,-0.324767,0.381641,2.843921e-02,0.972646,0.972646,0.009135,2.016150e-05,3.771759e-03
1126,2025-11-02,3976,2592000,1.644569e-04,-0.084006,0.616629,2.897839e-03,0.990929,0.990929,0.030446,9.802746e-06,1.266775e-03
1127,2025-11-02,4028,2592000,2.650676e-04,0.075606,0.053753,1.031218e-02,0.969083,0.969082,0.010477,1.283265e-05,2.583337e-03


In [30]:
set_data(
    engine,
    models.ProviderAssetGroupAttribute.__tablename__,
    toset,
    operation_type="upsert",
)

Upserting 1129 row(s) to provider_asset_group_attribute


In [17]:
cluster.close(force_shutdown=True)

[2025-11-03 18:52:57,433][INFO    ][coiled] Cluster 1242425 deleted successfully.
